# MP2 Part 1 — commit retrieval for aamphone's 10 projects
Retrieves every commit for each assigned project from World of Code (WoC V2412) and writes `aamphone_project_summary.csv` (semicolon-separated: project_wocid; commit_sha1; author; time; commit message).

In [8]:
## WoC data (World of Code API, retrieved 2026-10-08)
Number of commits, number of distinct author strings, and min/max author time (Unix seconds, with the UTC date).

| Project | Commits | Authors | Min time | Max time |
|---|---|---|---|---|
| arviz-devs_arviz | 9,492 | 395 | 1438170670 (2015-07-29) | 1761135327 (2025-10-22) |
| clindatsci_dicomweb-python-client | 518 | 39 | 1521487507 (2018-03-19) | 1757491841 (2025-09-10) |
| peal_vole | 595 | 7 | 1395418492 (2014-03-21) | 1757678214 (2025-09-12) |
| nbara_python-meegkit | 479 | 45 | 1227075592 (2008-11-19) | 1775048948 (2026-04-01) |
| flyaflya_causact | 457 | 21 | 1524067862 (2018-04-18) | 1757705466 (2025-09-12) |
| geolytix_xyz | 15,306 | 65 | 1512569316 (2017-12-06) | 1762277501 (2025-11-04) |
| genouest_biomaj-download | 505 | 19 | 1472208351 (2016-08-26) | 1750133502 (2025-06-17) |
| pressio_pressio | 4,282 | 52 | 1525121445 (2018-04-30) | 1757514162 (2025-09-10) |
| kkondo1981_aglm | 366 | 6 | 1546232994 (2018-12-31) | 1746979099 (2025-05-11) |
| winvector_vtreat | 1,081 | 6 | 1407432287 (2014-08-07) | 1736461287 (2025-01-09) |

33,140 unique commit SHA1s were listed by `p2c`; 59 returned "Key … not found" from WoC's commit store, so 33,081 are in `aamphone_project_summary.csv`.

**Validation against GitHub.** WoC counts are much higher than GitHub's default-branch counts (e.g., arviz 9,492 vs 1,670 on `main`; geolytix_xyz 15,306 vs 6,217) because WoC's `p2c` includes commits from all branches and from forks it attributes to the project. Author counts are also inflated because the same person appears under several name/e-mail strings.

In [9]:
import sys
sys.path.insert(0, 'woc_pkg')  # the WoC client unpacked by the cell above
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
import time

netid = 'aamphone'
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
# woc = WocMapsRemote(base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY")

# Assigned projects from net2prj.csv (netID == aamphone): WoC id -> GitHub URL
projects = {
    'arviz-devs_arviz': 'https://github.com/arviz-devs/arviz',
    'clindatsci_dicomweb-python-client': 'https://github.com/clindatsci/dicomweb-python-client',
    'peal_vole': 'https://github.com/peal/vole',
    'nbara_python-meegkit': 'https://github.com/nbara/python-meegkit',
    'flyaflya_causact': 'https://github.com/flyaflya/causact',
    'geolytix_xyz': 'https://github.com/GEOLYTIX/xyz',
    'genouest_biomaj-download': 'https://github.com/genouest/biomaj-download',
    'pressio_pressio': 'https://github.com/Pressio/pressio',
    'kkondo1981_aglm': 'https://github.com/kkondo1981/aglm',
    'winvector_vtreat': 'https://github.com/WinVector/vtreat',
}

## Step 1 — commit SHA1s per project (p2c map)

In [10]:
list_df_commits = []
for prj in projects:
    for attempt in range(3):
        try:
            commits = woc.get_values('p2c', prj)
            break
        except Exception as e:
            print(prj, 'attempt', attempt + 1, 'failed:', e)
            time.sleep(5)
    else:
        raise RuntimeError('Could not retrieve commits for ' + prj)
    d = pd.DataFrame(commits, columns=['sha1'])
    d['project'] = prj
    print(f'{prj}: {len(d)} commits')
    list_df_commits.append(d)
    time.sleep(1)

df = pd.concat(list_df_commits, ignore_index=True)
df.to_csv('df_commits.csv', index=False)  # saved so the SHA1 list need not be fetched again
df.groupby('project').size()

arviz-devs_arviz: 9515 commits
clindatsci_dicomweb-python-client: 518 commits
peal_vole: 595 commits
nbara_python-meegkit: 479 commits
flyaflya_causact: 457 commits
geolytix_xyz: 15306 commits
genouest_biomaj-download: 506 commits
pressio_pressio: 4284 commits
kkondo1981_aglm: 366 commits
winvector_vtreat: 1114 commits


,0
project,
arviz-devs_arviz,9515
clindatsci_dicomweb-python-client,518
flyaflya_causact,457
genouest_biomaj-download,506
geolytix_xyz,15306
kkondo1981_aglm,366
nbara_python-meegkit,479
peal_vole,595
pressio_pressio,4284


## Step 2 — commit content in batches
Batches of 10 SHA1s (below the 50 limit) with a 1-second wait between requests. Failed batches are retried, and any SHA1 still missing is reported at the end.

In [11]:
# a commit can belong to several projects, so retrieve each unique SHA1 once
shas = df['sha1'].drop_duplicates().tolist()
chunks = [shas[x:x + 10] for x in range(0, len(shas), 10)]
commit_content = {}
failed = []

for chunk in tqdm(chunks):
    for attempt in range(3):
        try:
            res, err = woc.get_values_many('commit.tch', chunk)
            break
        except Exception as e:
            print('retrying after error:', e)
            time.sleep(5 * (attempt + 1))
    else:
        failed.extend(chunk)
        continue
    if err:
        print('Got Errors', err)
    for k, v in res.items():
        commit_content[k] = v[0]  # needed because of the internal implementation
    time.sleep(1)  # stay under the rate limit

missing = [s for s in shas if s not in commit_content]
print('retrieved', len(commit_content), 'of', len(shas), 'unique commits; missing:', len(missing))

  2%|▏         | 58/3314 [01:00<56:27,  1.04s/it]

Got Errors {'0ebb3eb1ab86750b2d8825500e818fe0d09c1cda': 'Key 0ebb3eb1ab86750b2d8825500e818fe0d09c1cda not found in /da5_fast/All.sha1c/commit_14.tch'}


  3%|▎         | 92/3314 [01:36<56:14,  1.05s/it]

Got Errors {'17dc14ed8122bbc869ab1b953f00b4e56921f00c': 'Key 17dc14ed8122bbc869ab1b953f00b4e56921f00c not found in /da5_fast/All.sha1c/commit_23.tch'}


  6%|▌         | 195/3314 [03:23<54:08,  1.04s/it]

...

Got Errors {'f950f900921a3023381bd9d8998b74d39234040e': 'Key f950f900921a3023381bd9d8998b74d39234040e not found in /da5_fast/All.sha1c/commit_121.tch'}


100%|█████████▉| 3312/3314 [57:44<00:02,  1.04s/it]

Got Errors {'fc3857fba4ba2d3631095fd1c0ef9e03d89ab69a': 'Key fc3857fba4ba2d3631095fd1c0ef9e03d89ab69a not found in /da5_fast/All.sha1c/commit_124.tch'}


100%|██████████| 3314/3314 [57:46<00:00,  1.05s/it]

retrieved 33081 of 33140 unique commits; missing: 59


In [12]:
# flatten: commit = [tree, [parents], [author, unixtime, tz], [committer, unixtime, tz], message]
commit_data = []
for sha, commit in commit_content.items():
    commit_data.append({
        'commit': sha,
        'tree': commit[0],
        'parent': list(commit[1]),
        'author': commit[2][0],
        'author_time': int(commit[2][1]),
        'author_tz': commit[2][2],
        'committer': commit[3][0],
        'committer_time': int(commit[3][1]),
        'committer_tz': commit[3][2],
        'message': commit[4],
    })
df_commit_data = pd.DataFrame(commit_data).merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
print(commit_data[0])
df_commit_data.head(2)

{'commit': '0001654c779c4bffe0ab685c2b1fc498410fed61', 'tree': 'fae0a5eec0d4a33528115c2b3bcc107c58ff0ee4', 'parent': ['220a17cfa35a14e0ac3e9ce2b88728dcb34c09b7'], 'author': 'Piyush Gautam <gautampiyus@gmail.com>', 'author_time': 1581595380, 'author_tz': '+0530', 'committer': 'GitHub <noreply@github.com>', 'committer_time': 1581595380, 'committer_tz': '+0200', 'message': "Make the legends interactive in bokeh (#1024)\n\n* Made the legends interactive\r\n\r\n* Changed the position of legend to top\r\n\r\n* Added interactive legends to plots energy, ess, kde\r\n\r\n* linting changes\r\n\r\n* final changes\r\n\r\n* Minor changes\r\n\r\n* sphinx error correction\r\n\r\n* minor changes\r\n\r\n* Changes\r\n\r\n* corrected the sphinx error\r\n\r\n* fixed the return axes/glyph issue\r\n\r\n* fixed minor errors\r\n\r\n* added the change to changelog\r\n\r\n* corrected the change to changelog\r\n\r\n* removed the kwargs 'legend' and 'label' from bokeh backend\r\n\r\n* optimized the code using def

,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0001654c779c4bffe0ab685c2b1fc498410fed61,fae0a5eec0d4a33528115c2b3bcc107c58ff0ee4,[220a17cfa35a14e0ac3e9ce2b88728dcb34c09b7],Piyush Gautam <gautampiyus@gmail.com>,1581595380,+0530,GitHub <noreply@github.com>,1581595380,+0200,Make the legends interactive in bokeh (#1024)\...,0001654c779c4bffe0ab685c2b1fc498410fed61,arviz-devs_arviz
1,000336440e29a42381d1dd7bf37b8fe62ad9f1bc,18e365fd3fc6c32041665a0986912c7ecd99aaa3,[692e46efd58687076204852366c13e33c4bbc57a],Oriol (ZBook) <oriol.abril.pla@gmail.com>,1637788645,+0200,Oriol (ZBook) <oriol.abril.pla@gmail.com>,1639106173,+0200,always copy kwargs dict inplace\n,000336440e29a42381d1dd7bf37b8fe62ad9f1bc,arviz-devs_arviz


## Step 3 — write aamphone_project_summary.csv
One row per (project, commit). The project column comes from each commit's own project (the template reused the loop variable `prj`, which would label every row with the last project). The file is written once in `w` mode so re-running the cell does not duplicate rows.

In [13]:
dfinf = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']].rename(columns={'author_time': 'time'})
dfinf = dfinf.sort_values(['project', 'time']).reset_index(drop=True)
dfinf.to_csv(netid + '_project_summary.csv', index=False, sep=';', mode='w', header=False)
dfinf.head(1)

,project,commit,author,time,message
0,arviz-devs_arviz,9698e0a4595cc242ce571b2908adc78afdea738b,Thomas Wiecki <thomas.wiecki@gmail.com>,1438170670,Initial commit\n


## Step 4 — WoC statistics per project

In [14]:
woc_stats = dfinf.groupby('project').agg(
    ncommits=('commit', 'nunique'),
    nauthors=('author', 'nunique'),
    min_time=('time', 'min'),
    max_time=('time', 'max'),
)
woc_stats['from'] = pd.to_datetime(woc_stats['min_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats['to'] = pd.to_datetime(woc_stats['max_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats

,ncommits,nauthors,min_time,max_time,from,to
project,,,,,,
arviz-devs_arviz,9492,395,1438170670,1761135327,2015-07-29,2025-10-22
clindatsci_dicomweb-python-client,518,39,1521487507,1757491841,2018-03-19,2025-09-10
flyaflya_causact,457,21,1524067862,1757705466,2018-04-18,2025-09-12
genouest_biomaj-download,505,19,1472208351,1750133502,2016-08-26,2025-06-17
geolytix_xyz,15306,65,1512569316,1762277501,2017-12-06,2025-11-04
kkondo1981_aglm,366,6,1546232994,1746979099,2018-12-31,2025-05-11
nbara_python-meegkit,479,45,1227075592,1775048948,2008-11-19,2026-04-01
peal_vole,595,7,1395418492,1757678214,2014-03-21,2025-09-12
pressio_pressio,4282,52,1525121445,1757514162,2018-04-30,2025-09-10


## GitHub data (collected 2026-10-07)
Stars and forks as displayed on each repository page (GitHub rounds counts above 1,000, so arviz shows 1.8k). Last commit date is the newest commit on the default branch.

| Project | GitHub repo | Stars | Forks | Last commit date |
|---|---|---|---|---|
| arviz-devs_arviz | arviz-devs/arviz | 1.8k | 501 | 2026-09-09 |
| clindatsci_dicomweb-python-client | clindatsci/dicomweb-python-client (now redirects to ImagingDataCommons/dicomweb-client) | 129 | 42 | 2026-08-17 |
| peal_vole | peal/vole | 8 | 2 | 2026-07-01 |
| nbara_python-meegkit | nbara/python-meegkit | 227 | 57 | 2026-08-21 |
| flyaflya_causact | flyaflya/causact | 47 | 12 | 2025-09-12 |
| geolytix_xyz | GEOLYTIX/xyz | 132 | 44 | 2026-10-02 |
| genouest_biomaj-download | genouest/biomaj-download | 1 | 7 | 2025-05-20 |
| pressio_pressio | Pressio/pressio (default branch: develop) | 50 | 7 | 2025-09-10 |
| kkondo1981_aglm | kkondo1981/aglm | 17 | 5 | 2025-05-12 |
| winvector_vtreat | WinVector/vtreat | 284 | 45 | 2025-01-09 |


## WoC data
(Fill in from the `woc_stats` table above: number of commits, number of authors, min time, max time.)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

In [15]:
import pandas as pd
from google.colab import files

rows = []
for prj, sha in zip(df['project'], df['sha1']):
    c = commit_content.get(sha)
    if c is not None:
        rows.append((prj, sha, c[2][0], int(c[2][1]), c[4]))
summary = pd.DataFrame(rows).sort_values([0, 3])
summary.to_csv('aamphone_project_summary.csv', sep=';', header=False, index=False)
print(len(summary), 'rows written')
files.download('aamphone_project_summary.csv')

33081 rows written


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>